In [1]:
import pandas as pd 
import numpy as np
from datetime import datetime
from typing import Dict, List, Union, Optional

import mlflow
import mlflow.sklearn

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV, RandomizedSearchCV
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.base import BaseEstimator

from sklearn.metrics import make_scorer

from catboost import CatBoostRegressor
from xgboost import XGBRegressor

from trading_robot.data_collection.data_collector import DataCollector
from trading_robot.utils.logger import log_message
from trading_robot.ml_src.mlflow_deploy_menadger import ModelDeploymentManager

import warnings
warnings.filterwarnings("ignore")
from sklearn.metrics import r2_score

from sklearn.metrics import make_scorer, mean_squared_error, r2_score


In [ ]:

class TimeSeriesModelEvaluator(ModelDeploymentManager):
    """
    A class for evaluating models and selecting hyperparameters specifically for time series data.

    Attributes:
    ----------
    n_splits : int
        The number of folds for TimeSeriesSplit.
    scaler : StandardScaler
        An object for scaling features.

    Methods:
    -------
    timeseries_grid_search(model, param_grid, X, y, cv=5, model_name=None) -> GridSearchCV
        Searches for model hyperparameters using GridSearchCV for time series.

    tune_and_log_models(X, y, models=None, param_grids=None, cv=5, deploy=True) -> dict
        Iterates over models, selects hyperparameters, and logs the best models in MLflow with RMSE.
        Optionally deploys the best model to production.

    rmse(y_true, y_pred) -> float
        Calculates Root Mean Squared Error (RMSE).

    """

    __models = {
        'random_forest': RandomForestRegressor(random_state=42),
        'gbr': GradientBoostingRegressor(random_state=42),
        'xgboost': XGBRegressor(random_state=42),
        'catboost': CatBoostRegressor(random_state=42, silent=True),
        'svr': SVR(),
        'polynomial': Pipeline([
            ('poly', PolynomialFeatures()),
            ('linear', LinearRegression())
        ]),
        'linear': LinearRegression(),
        'ridge': Ridge(random_state=42, max_iter=15000),
        'lasso': Lasso(random_state=42, max_iter=15000),
        'knn': KNeighborsRegressor(),
        
    }

    __params = {
                'catboost': {
                    'depth': range(4, 11),  
                    'learning_rate': np.arange(0.01, 0.1, 0.01),
                    'iterations': range(100, 1100, 100),
                    'l2_leaf_reg': np.arange(1, 10, 2),
                    'border_count': range(32, 129, 32)
                },
                'xgboost': {
                    'n_estimators': range(100, 1100, 100),
                    'max_depth': range(3, 10, 2),
                    'learning_rate': np.arange(0.01, 0.1, 0.01),
                    'subsample': np.arange(0.5, 1.1, 0.1),
                    'colsample_bytree': np.arange(0.5, 1.1, 0.1),
                    'gamma': [0, 0.1, 0.2, 0.3],
                    'reg_alpha': [0, 0.01, 0.1, 1],
                    'reg_lambda': [1, 0.1, 0.01, 0]
                },
                'random_forest': {
                    'n_estimators': range(100, 1100, 100), 
                    'max_depth': range(1, 16, 2), 
                    'min_samples_split': range(2, 12, 2), 
                    'min_samples_leaf': range(2, 8, 2),
                    'max_features': ['auto', 'sqrt', 'log2'],
                    'bootstrap': [True, False]
                },
                'gbr': {
                    'n_estimators': range(100, 1100, 100),
                    'max_depth': range(1, 11, 2),
                    'subsample': np.arange(0.5, 1.1, 0.1),
                    'min_samples_split': range(2, 8, 2),
                    'min_samples_leaf': range(2, 8, 2),
                    'learning_rate': np.arange(0.1, 1.1, 0.1),
                    'max_features': ['auto', 'sqrt', 'log2'],
                    "criterion": ["friedman_mse", "mse", "mae"],
                    'loss': ['ls', 'lad', 'huber', 'quantile']
                },
                'svr': {
                    'C': [0.01, 0.1, 1, 10, 100, 1000],
                    'kernel': ['linear', 'poly', 'rbf', 'sigmoid'],
                    'gamma': ['scale', 'auto'],
                    'epsilon': [0.001, 0.01, 0.1, 1]
                },
                'polynomial': {
                    'poly__degree': range(2, 8, 2),  
                    'linear__fit_intercept': [True, False],
                    "poly__interaction_only": [True, False]
                },
                'ridge': {
                    'alpha': [0.1, 1.0, 10.0, 100.0],
                    "solver": ["auto", "svd", "cholesky", "lsqr", "sparse_cg", "sag", "saga"],
                    'fit_intercept': [True, False]
                },
                'lasso': {
                    'alpha': [0.0001, 0.001, 0.01, 0.1, 1.0, 10.0],
                    'fit_intercept': [True, False]
                },
                'knn': {
                    'n_neighbors': range(2, 17, 2),
                    'weights': ['uniform', 'distance'],
                    'p': [1, 2]  # 1: Manhattan, 2: Euclidean
                },
            }

    def __init__(self, scaler = None):
        """
        Initializes the TimeSeriesModelEvaluator with a StandardScaler for feature scaling.
        """
        super().__init__()
        self.scaler = scaler if scaler is not None else StandardScaler()

    def timeseries_param_search(self, model: BaseEstimator, 
                                param_grid: Dict[str, List[Union[float, int, str]]],
                                X: pd.DataFrame, y: pd.Series, search_model: str = "random", 
                                cv: int = 5, n_iter: int = 10, random_state: int = 42,
                                n_jobs=None):
        """
        Searches for the best hyperparameters for the model using GridSearchCV or RandomizedSearchCV with time series cross-validation.

        Parameters:
        ----------
        search_model : str, optional
            The type of hyperparameter search: "grid" for exhaustive search (GridSearchCV) or "random" for randomized search (RandomizedSearchCV).
            Defaults to "random".
        model : BaseEstimator
            A model instance for training (e.g., RandomForestRegressor).
        param_grid : Dict[str, List[Union[float, int, str]]]
            Dictionary with parameters to search through, where the keys are the parameter names and 
            the values are lists of parameter settings to try.
        X : pd.DataFrame
            DataFrame containing the features for training the model.
        y : pd.Series
            Series containing the target variable for training the model.
        cv : int, optional
            The number of folds for time series cross-validation (TimeSeriesSplit). Defaults to 5.
        n_iter : int, optional
            Number of iterations for randomized search. Only used when search_model="random". Defaults to 10.
        random_state : int, optional
            Seed for the random number generator. Defaults to None.

        Returns:
        -------
        param_search : GridSearchCV or RandomizedSearchCV
            The search object containing the best model parameters, scores, and other details about the cross-validation process.

        Notes:
        ------
        - TimeSeriesSplit is used for cross-validation, which is suitable for time series data where the order of observations is important.
        - A custom RMSE (Root Mean Square Error) scorer is used to evaluate the models.
        - If `mlflow_log_metrics` is set to True, additional metrics and details are logged to MLflow to track the model's performance with the selected parameters.

        Example:
        --------
        >>> model = RandomForestRegressor()
        >>> param_grid = {
        >>>     'n_estimators': [100, 200],
        >>>     'max_depth': [10, 20, None],
        >>> }
        >>> grid_search_result = timeseries_param_search(search_model="grid", model=model, param_grid=param_grid, X=X, y=y, cv=5)
        >>> print(grid_search_result.best_params_)
        """
        
        # Custom scorer based on RMSE
        scorer = make_scorer(self.rmse)

        # GridSearchCV for exhaustive parameter search
        if search_model == "grid":
            param_search = GridSearchCV(estimator=model, param_grid=param_grid, 
                                        cv=TimeSeriesSplit(n_splits=cv), 
                                        scoring=scorer, random_state=random_state,
                                        n_jobs=n_jobs)
        
        # RandomizedSearchCV for randomized parameter search
        elif search_model == "random":
            param_search = RandomizedSearchCV(estimator=model, param_distributions=param_grid, 
                                            n_iter=n_iter, cv=TimeSeriesSplit(n_splits=cv), 
                                            scoring=scorer, random_state=random_state,
                                            n_jobs=n_jobs)
            
        # Raise error if an unsupported search_model is provided
        else:
            raise KeyError("Unknown search model: use 'grid' or 'random'")

        # Fit the model to the data
        param_search.fit(X, y)

        # Log the best parameters and score
        log_message(f"Best parameters found: {param_search.best_params_}")
        log_message(f"Best score achieved: {param_search.best_score_}")
        
        return param_search
    
    def tune_and_log_models(self, X: pd.DataFrame, y: pd.Series,
                            X_test: pd.DataFrame, y_test: pd.Series, 
                            models : None | Dict[str, BaseEstimator] = None, 
                            param_grids : None | Dict[str, Dict[str, List[Union[float, int, str]]]] = None, 
                            cv=5, search_model: str = "random", random_state: int = 42,
                            n_iter: int = 10, n_jobs=None,
                            mlflow_log_metrics : bool = True, deploy=True):
        """
            Tunes hyperparameters for multiple models, logs them and their metrics to MLflow, 
            and optionally deploys the best model.

            Parameters:
            ----------
            X : pd.DataFrame
                DataFrame containing the features used for training the models.
            y : pd.Series
                Series containing the target variable for training the models.
            X_test : pd.DataFrame
                DataFrame containing the features used for testing the models.
            y_test : pd.Series
                Series containing the target variable for testing the models.
            search_model : str, optional
                    Type of hyperparameter search: "grid" for exhaustive search (GridSearchCV) or 
                    "random" for random search (RandomizedSearchCV). Default is "random".
            models : dict, optional
                A dictionary where keys are model names and values are instances of the models 
                (e.g., {'ridge': Ridge(), 'lasso': Lasso()}). Default is None, which will use predefined models.
            param_grids : dict, optional
                A dictionary where keys are model names and values are parameter grids for those models. 
                Each parameter grid is a dictionary where the keys are parameter names and the values are lists of parameter values to try. 
                Default is None, which will use predefined parameter grids.
            cv : int, optional
                The number of folds for cross-validation. Default is 5.
            n_iter : int, optional
                    Number of iterations for random search. Used only if search_model="random". Default is 10.
            random_state : int, optional
                Seed for the random number generator. Default is None.
            n_jobs : int, optional
                Number of jobs to run in parallel. -1 means using all processors. Default is -1.

            mlflow_log_metrics : bool, optional
                If True, logs additional metrics and model parameters to MLflow for tracking. Default is True.
            deploy : bool, optional
                If True, the best model is automatically deployed after hyperparameter tuning and model selection. 
                Default is True.

            Returns:
            -------
            dict
                A dictionary where keys are model names and values are dictionaries containing the RMSE values 
                ('rmse' key) for the corresponding models.

            Notes:
            ------
            - This method iterates over the provided models, applies Grid Search with TimeSeriesSplit for cross-validation, 
            and logs the best model for each algorithm to MLflow.
            - The best model across all candidates, based on RMSE, can be automatically deployed if `deploy` is set to True.
            - The method also handles data preprocessing, including scaling of features, to ensure consistency 
            during the model training and evaluation processes.
            - The example input data is captured from the last row of the scaled features, which is used in the model registration process.

            Example:
            --------
            >>> from sklearn.linear_model import Ridge, Lasso
            >>> model_dict = {'ridge': Ridge(), 'lasso': Lasso()}
            >>> param_grid_dict = {
            >>>     'ridge': {'alpha': [0.1, 1.0, 10.0]},
            >>>     'lasso': {'alpha': [0.001, 0.01, 0.1]}
            >>> }
            >>> results = tune_and_log_models(X_train, y_train, X_test, y_test, models=model_dict, param_grids=param_grid_dict, cv=5, deploy=True)
            >>> print(results)
            """
        if models is None:
            models = self.__models

        if param_grids is None:
            param_grids = self.__params

        X = self.__transform(X)

        results = {}
        selected_model_rmse = np.inf
        selected_model = None
        selected_model_name = None
        run_id = None

        parent_run_name = f"Model_Tuning_{datetime.now().strftime('%Y-%m-%d_%H-%M')}"
        log_message(f"Starting model tuning with parent run name: {parent_run_name}")

        with mlflow.start_run(run_name=parent_run_name) as parent_run:
            for model_name, candidate_model in models.items():
                with mlflow.start_run(run_name=model_name, nested=True):

                    log_message(f"Starting grid search with TimeSeriesSplit for model '{model_name}'.")

                    # Get the parameter grid for the current model
                    param_grid = param_grids.get(model_name, {})

                    # Grid Search with TimeSeriesSplit
                    grid_search = self.timeseries_param_search(
                        model=candidate_model,
                        param_grid=param_grid,
                        X=X,
                        y=y,
                        cv=cv,
                        random_state=random_state,
                        n_iter=n_iter,
                        search_model=search_model,
                        n_jobs=n_jobs
                    )

                    # Best model after Grid Search
                    best_model = grid_search.best_estimator_

                    # Scale test data similarly to training data
                    X_test = pd.DataFrame(self.scaler.fit_transform(X_test), columns=X_test.columns)
                    
                    # Predict on test data
                    predict = best_model.predict(X_test)  

                    # Calculate RMSE for predictions on test data
                    rmse = self.rmse(y_test, predict)

                    # Store RMSE results for the current model
                    results[model_name] = {'rmse': rmse}

                    # Select the model with the lowest RMSE
                    if rmse < selected_model_rmse:
                        selected_model_rmse = rmse
                        selected_model = best_model
                        selected_model_name = model_name
                        active_run = mlflow.active_run()
                        run_id = active_run.info.run_id

                    # Log additional metrics and data
                    if mlflow_log_metrics:
                        log_message(f"Logging additional metrics to MLflow.")
                        self._log_additional_metrics(grid_search, X_test, y_test, model_name, self.scaler)

            # Register and deploy the best model if required
            if deploy and selected_model is not None:
                print("*" * 100)
                example_input = X.tail(1).to_dict(orient='records')[0]


                log_message(f"Deploying the best model: {selected_model_name}")
                # The best model should be registered and deployed
                # Here we assume `selected_model` has the necessary attributes to register
                self.register_and_deploy_best_model(run_id=run_id, best_model=selected_model, model_name=selected_model_name, 
                                                    example_input=example_input, scaler= self.scaler )

        log_message("Model tuning and logging complete")
        return results

    def select_top_n_models(self, X_train: pd.DataFrame, y_train: pd.Series,
                            X_test: pd.DataFrame, y_test: pd.Series,
                            models: Dict[str, BaseEstimator] = None, n_top: int = 3,
                            return_names_only: bool = False, n_splits: int = 5) -> Union[Dict[str, Dict], List[str]]:
        """
        Trains and evaluates multiple models from the given dictionary and selects the top N models
        based on their performance on the test data, including cross-validation using TimeSeriesSplit
        for time series data.

        Parameters:
        ----------
        X_train : pd.DataFrame
            DataFrame containing the features used for training the models.
        y_train : pd.Series
            Series containing the target variable for training the models.
        X_test : pd.DataFrame
            DataFrame containing the features used for testing the models.
        y_test : pd.Series
            Series containing the target variable for testing the models.
        models : dict
            A dictionary where keys are model names and values are instances of the models 
            (e.g., {'ridge': Ridge(), 'lasso': Lasso()}).
        n_top : int, optional
            The number of top models to return. Default is 3.
        return_names_only : bool, optional
            If True, return only the list of model names. If False, return the dictionary with models and RMSE scores.
        n_splits : int, optional
            Number of time-based splits for cross-validation. Default is 5.

        Returns:
        -------
        Union[Dict[str, Dict], List[str]]
            If return_names_only is False, returns a dictionary where keys are model names and values are dictionaries containing 
            the trained model, the RMSE score, the R^2 score, and the cross-validation RMSE score.
            If return_names_only is True, returns a list of model names.
        """
        if models is None:
            models = self.__models

        # Dictionary to store model performance results
        results = {}

        # TimeSeriesSplit for cross-validation
        tscv = TimeSeriesSplit(n_splits=n_splits)

        # Iterate over each model in the provided dictionary
        for model_name, model in models.items():
            # List to store RMSE scores for each fold
            cv_rmse_scores = []

            # Perform TimeSeriesSplit cross-validation
            for train_idx, val_idx in tscv.split(X_train):
                X_train_fold, X_val_fold = X_train.iloc[train_idx], X_train.iloc[val_idx]
                y_train_fold, y_val_fold = y_train.iloc[train_idx], y_train.iloc[val_idx]

                # Train the model on the training fold
                model.fit(X_train_fold, y_train_fold)

                # Predict on the validation fold
                y_val_pred = model.predict(X_val_fold)

                # Calculate RMSE for the validation fold and store it
                rmse_fold = self.rmse(y_val_fold, y_val_pred)
                cv_rmse_scores.append(rmse_fold)

            # Calculate the mean RMSE across all validation folds
            mean_cv_rmse = np.mean(cv_rmse_scores)

            # Train the model on the full training data
            model.fit(X_train, y_train)

            # Predict the target values on the test data
            y_pred = model.predict(X_test)

            # Calculate RMSE and R^2 for the model's predictions on test data
            rmse = self.rmse(y_test, y_pred)
            r2 = r2_score(y_test, y_pred)

            # Store the model, its RMSE, R^2, and cross-validation RMSE score
            results[model_name] = {
                'model': model,
                'rmse_test': rmse,
                'r2_test': r2,
                'cv_rmse': mean_cv_rmse
            }

        # Sort the models by their test RMSE in ascending order (lower RMSE is better)
        sorted_results = dict(sorted(results.items(), key=lambda item: item[1]['rmse_test']))

        # Select the top N models based on test RMSE
        top_n_models = dict(list(sorted_results.items())[:n_top])

        # Return either the model names or the full dictionary with scores
        if return_names_only:
            return list(top_n_models.keys())
        else:
            return top_n_models

    def __is_fitted(self, transformer: BaseEstimator):
        """
        The function checks if the transformer from Scikit-learn is trained.

        :param transformer: transformer object (e.g. StandardScaler, MinMaxScaler, PCA)
        :return: True if the transformer is trained, otherwise False
        """
        fitted_attributes = {
            'StandardScaler': 'n_samples_seen_',
            'MinMaxScaler': 'min_',
            'MaxAbsScaler': 'scale_',
            'RobustScaler': 'center_',
            'Normalizer': 'n_features_in_',  
            'PCA': 'components_',
            'KernelPCA': 'dual_coef_',
            'IncrementalPCA': 'components_',
            'TruncatedSVD': 'components_',
            'NMF': 'components_',
            'FactorAnalysis': 'components_',
            'DictionaryLearning': 'components_',
            'FastICA': 'components_',
            'GaussianRandomProjection': 'components_',
            'SparseRandomProjection': 'components_',
            'Binarizer': 'threshold',  
            'QuantileTransformer': 'n_quantiles_',
            'PowerTransformer': 'lambdas_',
            'FunctionTransformer': 'n_features_in_',  
        }
        
        transformer_name = transformer.__class__.__name__
        attribute = fitted_attributes.get(transformer_name)
        
        if attribute:
            return hasattr(transformer, attribute)
        else:
            raise ValueError(f"Unknown transformer: {transformer_name}")
        
    def __transform(self, X:  pd.DataFrame):
        """
        Applies scaling to the input data if the scaler has not been fitted.
        
        This method ensures that all data types in the DataFrame are converted to float64 to
        avoid issues with NaN values in integer columns. If the scaler has already been fitted, 
        it simply returns the input data as is. Otherwise, it fits the scaler to the input data, 
        scales the data, and returns the scaled DataFrame with the original column names.
        
        :param X: pd.DataFrame - The input data to be transformed.
        :return: pd.DataFrame - The transformed (scaled) data.
        """
        
        # Convert data types to avoid issues with NaN in integer columns
        X = X.astype(np.float64)

        # Check if the scaler has already been fitted
        if self.__is_fitted(self.scaler):
            # If the scaler is fitted, return the data as is
            X_scaled = X
        else:
            # If the scaler is not fitted, fit and transform the data
            # Convert the scaled data back into a DataFrame, retaining original column names
            X_scaled = pd.DataFrame(self.scaler.fit_transform(X), columns=X.columns)

        # Return the scaled data
        return X_scaled


models = {
    'random_forest': RandomForestRegressor(random_state=42),
    'gbr': GradientBoostingRegressor(random_state=42),
    'xgboost': XGBRegressor(random_state=42),
    'catboost': CatBoostRegressor(random_state=42, silent=True),
    'svr': SVR(),
    'polynomial': Pipeline([
    ('poly', PolynomialFeatures()),
    ('linear', LinearRegression())
    ]),
    'linear': LinearRegression(),
    'ridge': Ridge(random_state=42, max_iter=15000),
    'lasso': Lasso(random_state=42, max_iter=15000),
    'knn': KNeighborsRegressor(),
}

params = {
    'catboost': {
        'depth': range(4, 11),  
        'learning_rate': np.arange(0.01, 0.1, 0.01),
        'iterations': range(100, 1100, 100),
        'l2_leaf_reg': np.arange(1, 10, 2),
        'border_count': range(32, 129, 32)
    },
    'xgboost': {
        'n_estimators': range(100, 1100, 100),
        'max_depth': range(3, 10, 2),
        'learning_rate': np.arange(0.01, 0.1, 0.01),
        'subsample': np.arange(0.5, 1.1, 0.1),
        'colsample_bytree': np.arange(0.5, 1.1, 0.1),
        'gamma': [0, 0.1, 0.2, 0.3],
        'reg_alpha': [0, 0.01, 0.1, 1],
        'reg_lambda': [1, 0.1, 0.01, 0]
    },
    'random_forest': {
        'n_estimators': range(100, 1100, 100), 
        'max_depth': range(1, 16, 2), 
        'min_samples_split': range(2, 12, 2), 
        'min_samples_leaf': range(2, 8, 2),
        'max_features': ['auto', 'sqrt', 'log2'],
        'bootstrap': [True, False]
    },
    'gbr': {
        'n_estimators': range(100, 1100, 100),
        'max_depth': range(1, 11, 2),
        'subsample': np.arange(0.5, 1.1, 0.1),
        'min_samples_split': range(2, 8, 2),
        'min_samples_leaf': range(2, 8, 2),
        'learning_rate': np.arange(0.1, 1.1, 0.1),
        'max_features': ['auto', 'sqrt', 'log2'],
        "criterion": ["friedman_mse", "mse", "mae"],
        'loss': ['ls', 'lad', 'huber', 'quantile']
    },
    'svr': {
        'C': [0.01, 0.1, 1, 10, 100, 1000],
        'kernel': ['linear', 'poly', 'rbf', 'sigmoid'],
        'gamma': ['scale', 'auto'],
        'epsilon': [0.001, 0.01, 0.1, 1]
    },
    'polynomial': {
        'poly__degree': range(2, 8, 2),  
        'linear__fit_intercept': [True, False],
        "poly__interaction_only": [True, False]
    },
    'ridge': {
        'alpha': [0.1, 1.0, 10.0, 100.0],
        "solver": ["auto", "svd", "cholesky", "lsqr", "sparse_cg", "sag", "saga"],
        'fit_intercept': [True, False]
    },
    'lasso': {
        'alpha': [0.0001, 0.001, 0.01, 0.1, 1.0, 10.0],
        'fit_intercept': [True, False]
    },
    'knn': {
        'n_neighbors': range(2, 17, 2),
        'weights': ['uniform', 'distance'],
        'p': [1, 2]  # 1: Manhattan, 2: Euclidean
    },
}


In [ ]:
data_col = DataCollector()

symbols = ["EURUSD", "AUDCAD", "GBPUSD"]

all_X_train, all_y_train, all_X_test, all_y_test = pd.DataFrame(), pd.Series(), pd.DataFrame(), pd.Series()

for symbol in symbols:
    data = data_col.get_historical_data(symbol=symbol)

    X_scaler = StandardScaler()
    y_scaler = StandardScaler()

    # Define split point (e.g., 80% of the data for training and 20% for testing)
    split_index = int(len(data) * 0.8)  # Calculate the index to split the data

    # Split the data into training and testing sets
    train = data.iloc[:split_index]
    test = data.iloc[split_index:]

    # Separate features (X) and target variable (y) from the dataset
    X_train = train.drop(columns="Close")
    y_train = train["Close"]

    # Save original data for comparison
    original_y_train = y_train.copy()

    # Scale features and target
    X_train_scaled = pd.DataFrame(X_scaler.fit_transform(X_train), columns=X_train.columns)
    y_train_scaled = pd.Series(y_scaler.fit_transform(y_train.values.reshape(-1, 1)).flatten(), index=y_train.index)

    # Repeat for test data
    X_test = test.drop(columns="Close")
    y_test = test["Close"]

    X_test_scaled = pd.DataFrame(X_scaler.transform(X_test), columns=X_test.columns)
    y_test_scaled = pd.Series(y_scaler.transform(y_test.values.reshape(-1, 1)).flatten(), index=y_test.index)


# ---- ПЕРЕД КОНКАТЕНАЦИЕЙ ----
    print(f"\n=== Symbol: {symbol} ===")
    print(f"Before concatenation (all_X_train): Shape: {all_X_train.shape}, Mean: {all_X_train.mean().mean()}, Std: {all_X_train.std().mean()}")
    print(f"Before concatenation (all_y_train): Shape: {all_y_train.shape}, Mean: {all_y_train.mean()}, Std: {all_y_train.std()}")
    
    # Concatenate scaled data
    all_X_train = pd.concat([all_X_train, X_train_scaled], ignore_index=True)
    all_y_train = pd.concat([all_y_train, y_train_scaled], ignore_index=True)
    all_X_test = pd.concat([all_X_test, X_test_scaled], ignore_index=True)
    all_y_test = pd.concat([all_y_test, y_test_scaled], ignore_index=True)

    # ---- ПОСЛЕ КОНКАТЕНАЦИИ ----
    print(f"After concatenation (all_X_train): Shape: {all_X_train.shape}, Mean: {all_X_train.mean().mean()}, Std: {all_X_train.std().mean()}")
    print(f"After concatenation (all_y_train): Shape: {all_y_train.shape}, Mean: {all_y_train.mean()}, Std: {all_y_train.std()}")

# Итоговая информация о данных после обработки всех символов
print("\n=== Final Combined Data ===")
print(f"Final all_X_train: Shape: {all_X_train.shape}, Mean: {all_X_train.mean().mean()}, Std: {all_X_train.std().mean()}")
print(f"Final all_y_train: Shape: {all_y_train.shape}, Mean: {all_y_train.mean()}, Std: {all_y_train.std()}")


In [ ]:
# Create an instance of TimeSeriesModelEvaluator to handle model evaluation and tuning
tsem = TimeSeriesModelEvaluator()

select_models =  tsem.select_top_n_models(
                                    X_train=all_X_train, 
                                    y_train=all_y_train,
                                    X_test=all_X_test, 
                                    y_test=all_y_test, 
                                    models=models,
                                    n_top=5
                                )



In [ ]:
pd.DataFrame(select_models).T